# One Hot Encoding!!!

___
### Importing Libraries

In [ ]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('cars.csv')

In [3]:
df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [4]:
df['owner'].value_counts()

owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: int64

In [5]:
df['fuel'].value_counts()

fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64

___
## 1. OneHotEncoding using Pandas

pd.get_dummies() is used to perform One-Hot Encoding on categorical columns.

In this code, the fuel and owner columns are converted into separate binary columns. Each unique category becomes a new column containing 0 or 1.

For example, if the fuel column contains Petrol, Diesel,LPG and CNG, four separate columns will be created: Petrol, Diesel,LPG and CNG.

A value of 1 indicates that the row belongs to that category, while 0 indicates that it does not.

In [6]:
pd.get_dummies(df,columns=['fuel','owner'])

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,False,True,False,False,True,False,False,False,False
1,Skoda,120000,370000,False,True,False,False,False,False,True,False,False
2,Honda,140000,158000,False,False,False,True,False,False,False,False,True
3,Hyundai,127000,225000,False,True,False,False,True,False,False,False,False
4,Maruti,120000,130000,False,False,False,True,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,False,True,True,False,False,False,False
8124,Hyundai,119000,135000,False,True,False,False,False,True,False,False,False
8125,Maruti,120000,382000,False,True,False,False,True,False,False,False,False
8126,Tata,25000,290000,False,True,False,False,True,False,False,False,False


___
### One-Hot Encoding with drop_first

In this code, the fuel and owner columns are encoded, and drop_first=True is used to remove the first category from each encoded column.

This helps avoid Dummy Variable Trap by creating unnecessary redundant columns.

For example, if a column has three categories, One-Hot Encoding normally creates three columns. With drop_first=True, only two columns are created


In [7]:
pd.get_dummies(df,columns=['fuel','owner'],drop_first=True)

,brand,km_driven,selling_price,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,True,False,False,False,False,False,False
1,Skoda,120000,370000,True,False,False,False,True,False,False
2,Honda,140000,158000,False,False,True,False,False,False,True
3,Hyundai,127000,225000,True,False,False,False,False,False,False
4,Maruti,120000,130000,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8123,Hyundai,110000,320000,False,False,True,False,False,False,False
8124,Hyundai,119000,135000,True,False,False,True,False,False,False
8125,Maruti,120000,382000,True,False,False,False,False,False,False
8126,Tata,25000,290000,True,False,False,False,False,False,False


___
## 2. OneHotEncoding using Sklearn

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,0:4],df.iloc[:,-1],test_size=0.2,random_state=2)

In [9]:
X_train.head()

,brand,km_driven,fuel,owner
5571,Hyundai,35000,Diesel,First Owner
2038,Jeep,60000,Diesel,First Owner
2957,Hyundai,25000,Petrol,First Owner
7618,Mahindra,130000,Diesel,Second Owner
6684,Hyundai,155000,Diesel,First Owner


___
### Importing One Hot Encoder

OneHotEncoder is used to convert nominal categorical data into numerical values using One-Hot Encoding.

It creates a separate binary column for each unique category. Each row gets a value of 1 for the category it belongs to and 0 for all other categories.

In [10]:
from sklearn.preprocessing import OneHotEncoder

___
### Creating the OneHotEncoder Object

**drop='first'** removes the first category from each categorical column to avoid creating redundant columns.

**sparse_output=False** ,makes the encoder return a regular NumPy array instead of a sparse matrix.

**dtype=np.int32** specifies that the encoded values should be stored as 32-bit integers instead of float as it takes large storage.

In [12]:
ohe = OneHotEncoder(drop='first',sparse_output=False,dtype=np.int32)

In [13]:
X_train_new = ohe.fit_transform(X_train[['fuel','owner']])

In [14]:
X_test_new = ohe.transform(X_test[['fuel','owner']])

In [15]:
X_train_new.shape

(6502, 7)

___
### Combining Original and Encoded Columns

np.hstack() is used to combine arrays horizontally, meaning the columns are placed side by side.

In this code, the original brand and km_driven columns are first selected from X_train and converted into a NumPy array using values.

X_train_new contains the newly encoded categorical columns.

np.hstack() combines both arrays so that the original columns and the encoded columns are present together in one NumPy array.

This is useful when we want to combine selected original numerical columns with the newly encoded categorical features.

In [16]:
np.hstack((X_train[['brand','km_driven']].values,X_train_new))

array([['Hyundai', 35000, 1, ..., 0, 0, 0],
       ['Jeep', 60000, 1, ..., 0, 0, 0],
       ['Hyundai', 25000, 0, ..., 0, 0, 0],
       ...,
       ['Tata', 15000, 0, ..., 0, 0, 0],
       ['Maruti', 32500, 1, ..., 1, 0, 0],
       ['Isuzu', 121000, 1, ..., 0, 0, 0]], shape=(6502, 9), dtype=object)

___
### Getting the Encoded Feature Names

ohe.get_feature_names_out() is used to get the names of the new columns created by OneHotEncoder.

It returns the feature names for all the encoded categories in the same order as the encoded values in dataframe.

In [25]:
ohe.get_feature_names_out()

array(['fuel_Diesel', 'fuel_LPG', 'fuel_Petrol',
       'owner_Fourth & Above Owner', 'owner_Second Owner',
       'owner_Test Drive Car', 'owner_Third Owner'], dtype=object)

___
### Converting the Combined Array into a DataFrame

The combined NumPy array created using np.hstack() is converted back into a Pandas DataFrame using pd.DataFrame().

The columns parameter is used to assign meaningful column names to the DataFrame. It includes the original brand and km_driven columns along with the newly created encoded columns for fuel and owner.

In [26]:
X_train_final = pd.DataFrame(
    np.hstack((X_train[['brand','km_driven']].values, X_train_new)),
    columns=['brand', 'km_driven','fuel_Diesel', 'fuel_LPG', 'fuel_Petrol',	'owner_Fourth & Above Owner',	'owner_Second Owner',	'owner_Test Drive Car',	'owner_Third Owner']
)

X_train_final.head()

,brand,km_driven,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Hyundai,35000,1,0,0,0,0,0,0
1,Jeep,60000,1,0,0,0,0,0,0
2,Hyundai,25000,0,0,1,0,0,0,0
3,Mahindra,130000,1,0,0,0,1,0,0
4,Hyundai,155000,1,0,0,0,0,0,0


___
## One Hot Encoding with Most Frequent Categories

___
### Checking the Number of Unique Categories

df['brand'].nunique() is used to find the total number of unique brands present in the brand column.

There are 32 unique brands/categories in this Brand column.

In [28]:
df['brand'].nunique()

32

In [31]:
df['brand'].value_counts()

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Ambassador          4
Kia                 4
MG                  3
Daewoo              3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64

___
### Counting Category Frequency and Setting the Frequency Threshold

df['brand'].value_counts() is used to count how many times each unique brand appears in the brand column.

The result shows each brand along with its frequency, usually arranged from the most frequent brand to the least frequent brand.

The result is stored in the variable counts.

Counts contains a **Pandas Series** with **brand names as its index** and their **frequencies as the values**.
___
**threshold = 100** sets the minimum frequency required for a category to be considered a frequent category.

In this case, any brand that appears 100 or more times can be treated as a frequent category, while brands appearing fewer than 100 times can later be grouped into a common category.

The threshold value can be adjusted depending on the dataset and the number of categories.


In [ ]:
counts = df['brand'].value_counts()
threshold=100

___
### Identifying Low-Frequency Categories

**repl = counts[counts <= threshold].index ** ---> Find all brands whose frequency is 100 or less, extract their names, and store those names in repl.

___

1. **counts <= threshold** ---> check each brand value pair like this:

500 <= 100 → False
300 <= 100 → False
40 <= 100  → True
20 <= 100  → True

The important point is that Pandas Series comparisons automatically operate on the values so we don't need to write "counts.values"

So we get a Boolean Series:

Toyota    False
Honda     False
Suzuki    False
BMW        True
Audi       True
Kia        True
___

2. **counts[counts <= threshold]**

Now this part means : counts[condition]

which means:

"From counts, give me only the rows where the condition is True."

Therefore, we get:

BMW     80
Audi    40
Kia     20

Notice that we still have both the brand names and their frequencies.
___

3. **.index**

Now we use: .index

This tells Pandas:

"I only want the index of these selected rows."

In our case, the index contains the brand names.

So we get:

BMW
Audi
Kia
___

4. **repl =**

repl =

stores those brand names in the variable repl.

So the complete meaning is:

Find all brands whose frequency is 100 or less, extract their names, and store those names in repl.

In [21]:
repl = counts[counts <= threshold].index

___
### Handling Rare Categories

Pandas looks at each value (brand name) in the df['brand'] Series and checks whether that value matches any of the elements inside repl.
___

For example, suppose:

repl = ['BMW', 'Audi', 'Kia']

and your brand column is:

[Toyota, BMW, Honda, Audi, Kia, Toyota]
___
Pandas effectively checks:

Toyota → Is Toyota in repl? → No  → Keep Toyota  
BMW    → Is BMW in repl?    → Yes → Replace with uncommon  
Honda  → Is Honda in repl?  → No  → Keep Honda  
Audi   → Is Audi in repl?   → Yes → Replace with uncommon  
Kia    → Is Kia in repl?    → Yes → Replace with uncommon  
Toyota → Is Toyota in repl? → No  → Keep Toyota  
___
So the result becomes:

Toyota
uncommon
Honda
uncommon
uncommon
Toyota

In [22]:
pd.get_dummies(df['brand'].replace(repl, 'uncommon')).sample(5)

,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,uncommon
2087,False,False,False,False,False,False,False,False,False,True,False,False,False
4669,False,False,False,False,False,True,False,False,False,False,False,False,False
3227,False,False,False,False,False,False,False,False,False,False,False,True,False
6781,False,False,False,False,True,False,False,False,False,False,False,False,False
5489,False,True,False,False,False,False,False,False,False,False,False,False,False


___
### Thats all from One Hot Encoding Section !!!
___